<a href="https://colab.research.google.com/github/Nikhil4002-50-82/Automatic-Pedicle-Screw-Planning/blob/main/AutomaticPedicleScrewPlanning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# ***Phase 1- Geometry Based Planning***

In [1]:
!pip install TotalSegmentator

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.9/212.9 kB 7.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

***Algorithm***

This program automatically finds the best and safest pedicle screw paths in the spine using CT scan data and segmentation labels.

First, the program loads the CT scan and the segmented spine. Each vertebra (L1–L5) is identified and small noisy parts are removed so only real vertebrae remain.

Next, the program studies the shape and orientation of each vertebra to understand which direction is up-down, left-right, and front-back. This helps the program place screws correctly even if the spine is tilted.

Then, it finds the pedicle centers on the left and right sides. These are the thickest parts of bone where screws are usually inserted.

After that, the program finds the entry point on the back surface of the vertebra and tries many possible screw directions and diameters.

For each possible screw, the program checks:

* If the screw stays completely inside the bone
* If the screw is long enough
* How much bone surrounds the screw (safety margin)

Finally, the program chooses the best screw for each side of each vertebra based on:

* Safety (thicker bone is safer)
* Screw length (longer is stronger)
* Good alignment

The result is a list of safe pedicle screw sizes, lengths, and safety margins for vertebrae L1 to L5.

In [8]:
import numpy as np
import nibabel as nib
from scipy.ndimage import distance_transform_edt, label as cc_label
from scipy.ndimage import map_coordinates
from sklearn.decomposition import PCA

# Paths to CT scan and segmentation
ctPath="/content/drive/MyDrive/SpineData/case_001/case_0000.nii"
segPath="/content/drive/MyDrive/SpineData/case_001/spine_2-label.nii"

# Minimum voxels needed to accept a vertebra
voxelThreshold=5000

# Possible screw diameters in mm
globalDiameters=[8.5,7.5,7.0,6.5,5.5,5.0,4.5,4.0]

# Maximum diameter allowed per vertebra
maxDiameterPerLevel={
"L1":6.5,
"L2":7.0,
"L3":7.5,
"L4":8.5,
"L5":8.5
}

# Step size when moving along screw path
stepMM=0.5

# Minimum allowed screw length
minLengthMM=18

# Allowed angle search range
directionConeDegLR=35
directionConeDegSI=20

# Number of directions tested
directionSamplesLR=13
directionSamplesSI=9

# How we score screws
# Bigger DT = safer
# Longer screw = better
# Less tilt = better
wDT=5.0
wLen=1.5
wTilt=0.5

# Map segmentation labels to vertebra names
labelMap={5:"L1",4:"L2",3:"L3",2:"L4",1:"L5"}

# Load CT or segmentation file
def loadNifti(path):
    nii=nib.load(path)
    return nii.get_fdata(), nii.header.get_zooms(), nii.affine

ct,spacingCT,affineCT=loadNifti(ctPath)
seg,spacing,affine=loadNifti(segPath)

# Keep only large connected vertebra components
def getValidLabels(seg):
    valid=[]
    for labelVal in range(1,6):
        mask=(seg==labelVal)
        if np.sum(mask)==0:
            continue
        labeled,nc=cc_label(mask)
        sizes=np.bincount(labeled.ravel())
        sizes[0]=0
        largest=np.argmax(sizes)
        component=(labeled==largest)
        if np.sum(component)>voxelThreshold:
            valid.append((labelVal,component))
    return valid

validSegments=getValidLabels(seg)

# Find vertebra orientation using PCA
# Gives Up-Down, Left-Right, Front-Back axes
def computeStableFrame(mask,affine):
    coords=np.argwhere(mask)
    coordsWorld=nib.affines.apply_affine(affine,coords)
    centroid=coordsWorld.mean(axis=0)
    pca=PCA(n_components=3)
    pca.fit(coordsWorld-centroid)
    axes=pca.components_
    worldZ=np.array([0,0,1])
    siAxis=axes[np.argmax(np.abs(axes@worldZ))]
    if np.dot(siAxis,worldZ)<0:
        siAxis=-siAxis
    tempAxis=axes[np.argmin(np.abs(axes@worldZ))]
    lrAxis=tempAxis - np.dot(tempAxis,siAxis)*siAxis
    lrAxis/=np.linalg.norm(lrAxis)
    apAxis=np.cross(siAxis,lrAxis)
    apAxis/=np.linalg.norm(apAxis)
    return centroid,np.vstack([siAxis,lrAxis,apAxis])

# Distance from each voxel to bone boundary
# Big value = thick bone = safer
def computeDistance(mask):
    return distance_transform_edt(mask,sampling=spacing)

# Find left and right pedicle centers
# Choose thickest bone in pedicle area
def pedicleCenters(mask,dist,centroid,axes,affine):
    coords=np.argwhere(mask)
    coordsWorld=nib.affines.apply_affine(affine,coords)
    siAxis,lrAxis,apAxis=axes
    rel=coordsWorld-centroid
    siVals=rel@siAxis
    lrVals=rel@lrAxis
    apVals=rel@apAxis
    midMask=np.abs(siVals)<np.percentile(np.abs(siVals),35)
    posteriorMask=apVals<np.percentile(apVals,40)
    leftMask=lrVals<0
    rightMask=lrVals>0
    leftCoords=coords[midMask & posteriorMask & leftMask]
    rightCoords=coords[midMask & posteriorMask & rightMask]
    if len(leftCoords)<30 or len(rightCoords)<30:
        return None,None
    lVox=leftCoords[np.argmax(dist[leftCoords[:,0],leftCoords[:,1],leftCoords[:,2]])]
    rVox=rightCoords[np.argmax(dist[rightCoords[:,0],rightCoords[:,1],rightCoords[:,2]])]
    lMM=nib.affines.apply_affine(affine,lVox)
    rMM=nib.affines.apply_affine(affine,rVox)
    return lMM,rMM

# Find screw entry point on back surface
# Move backward from pedicle center until outside bone
def findEntry(center,axes,maskFloat,affine):
    siAxis,lrAxis,apAxis=axes
    direction=-apAxis
    invAff=np.linalg.inv(affine)
    p=center.copy()
    for _ in range(300):
        vox=nib.affines.apply_affine(invAff,p)
        if any(v<0 or v>=s-1 for v,s in zip(vox,maskFloat.shape)):
            break
        val=map_coordinates(maskFloat,[[vox[0]],[vox[1]],[vox[2]]],order=1)[0]
        if val<0.5:
            break
        p+=direction*stepMM
    return p+apAxis*1.0

# Check if screw cylinder stays inside bone
def cylinderSafe(p,d,radius,maskFloat,affine):
    invAff=np.linalg.inv(affine)
    for angle in np.linspace(0,2*np.pi,8,endpoint=False):
        offset=radius*(np.cos(angle)*np.cross(d,[0,0,1])+np.sin(angle)*np.cross(d,[1,0,0]))
        testPoint=p+offset
        vox=nib.affines.apply_affine(invAff,testPoint)
        if any(v<0 or v>=s-1 for v,s in zip(vox,maskFloat.shape)):
            return False
        val=map_coordinates(maskFloat,[[vox[0]],[vox[1]],[vox[2]]],order=1)[0]
        if val<0.5:
            return False
    return True

# Test one screw path
# Measure length and safety margin
def evaluate(entry,direction,maskFloat,dist,affine,radius,axes):
    d=direction/np.linalg.norm(direction)
    invAff=np.linalg.inv(affine)
    t=0
    minDT=999
    while True:
        if t<5:
            t+=stepMM
            continue
        p=entry+d*t
        vox=nib.affines.apply_affine(invAff,p)
        if any(v<0 or v>=s-1 for v,s in zip(vox,maskFloat.shape)):
            break
        maskVal=map_coordinates(maskFloat,[[vox[0]],[vox[1]],[vox[2]]],order=1)[0]
        if maskVal<0.5:
            break
        dtVal=map_coordinates(dist,[[vox[0]],[vox[1]],[vox[2]]],order=1)[0]
        minDT=min(minDT,dtVal)
        if not cylinderSafe(p,d,radius,maskFloat,affine):
            break
        t+=stepMM
    if t<minLengthMM:
        return None
    if minDT-radius<=0:
        return None
    siAxis=axes[0]
    tilt=abs(np.dot(d,siAxis))
    score=wDT*minDT + wLen*(t/10) - wTilt*tilt
    return score,t,minDT,p,d

# Try many diameters and directions
# Keep best safe screw
def optimize(center,axes,maskFloat,dist,affine,diameters):
    siAxis,lrAxis,apAxis=axes
    entry=findEntry(center,axes,maskFloat,affine)
    best=None
    for diam in diameters:
        radius=diam/2
        for lrAng in np.linspace(-directionConeDegLR,directionConeDegLR,directionSamplesLR):
            for siAng in np.linspace(-directionConeDegSI,directionConeDegSI,directionSamplesSI):
                direction=apAxis + np.tan(np.deg2rad(lrAng))*lrAxis + np.tan(np.deg2rad(siAng))*siAxis
                r=evaluate(entry,direction,maskFloat,dist,affine,radius,axes)
                if r is None:
                    continue
                score,length,minDT,tip,d=r
                if best is None or score>best[0]:
                    best=(score,entry,tip,length,minDT,diam)
    return best

print("\nGEOMETRY BASED PEDICLE SCREW PLANNER:\n")

for labelVal,mask in validSegments:
    name=labelMap.get(labelVal,str(labelVal))
    maxDiam=maxDiameterPerLevel[name]
    diameters=[d for d in globalDiameters if d<=maxDiam]
    print("==============")
    print(name)
    print("==============")
    print("Tested Diameters:",diameters)
    centroid,axes=computeStableFrame(mask,affine)
    dist=computeDistance(mask)
    maskFloat=mask.astype(np.float32)
    lCenter,rCenter=pedicleCenters(mask,dist,centroid,axes,affine)
    if lCenter is None:
        print("Pedicle detection failed\n")
        continue
    for side,center in [("Left",lCenter),("Right",rCenter)]:
        result=optimize(center,axes,maskFloat,dist,affine,diameters)
        if result is None:
            print(f"{side}: NO SAFE PATH\n")
            continue
        score,entry,tip,length,minDT,diam=result
        print(f"{side} Screw Found")
        print("Diameter:",diam,"mm")
        print("Length:",round(length,1),"mm")
        print("Safety Margin:",round(minDT-diam/2,2),"mm")
        print()


GEOMETRY BASED PEDICLE SCREW PLANNER:

L5
Tested Diameters: [8.5, 7.5, 7.0, 6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 6.5 mm
Length: 30.0 mm
Safety Margin: 0.44 mm

Right Screw Found
Diameter: 8.5 mm
Length: 45.0 mm
Safety Margin: 0.23 mm

L4
Tested Diameters: [8.5, 7.5, 7.0, 6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 6.5 mm
Length: 33.0 mm
Safety Margin: 0.63 mm

Right Screw Found
Diameter: 8.5 mm
Length: 39.5 mm
Safety Margin: 0.1 mm

L3
Tested Diameters: [7.5, 7.0, 6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 5.5 mm
Length: 52.0 mm
Safety Margin: 0.54 mm

Right Screw Found
Diameter: 5.0 mm
Length: 56.0 mm
Safety Margin: 0.62 mm

L2
Tested Diameters: [7.0, 6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 6.5 mm
Length: 45.5 mm
Safety Margin: 0.39 mm

Right Screw Found
Diameter: 6.5 mm
Length: 40.5 mm
Safety Margin: 0.45 mm

L1
Tested Diameters: [6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 6.5 mm
Length: 43.5 mm
Safety Margin: 0.27 mm

Right Screw

***Algorithm that supports visualization***

In [9]:
import numpy as np
import nibabel as nib
from scipy.ndimage import distance_transform_edt, label as cc_label
from scipy.ndimage import map_coordinates
from sklearn.decomposition import PCA

# Paths to CT scan and segmentation
ctPath="/content/drive/MyDrive/SpineData/case_002/case_0000.nii"
segPath="/content/drive/MyDrive/SpineData/case_002/spine_2-label.nii"

# Minimum voxels needed to accept a vertebra
voxelThreshold=5000

# Possible screw diameters (mm)
globalDiameters=[8.5,7.5,7.0,6.5,5.5,5.0,4.5,4.0]

# Maximum screw size allowed per vertebra
maxDiameterPerLevel={
"L1":6.5,
"L2":7.0,
"L3":7.5,
"L4":8.5,
"L5":8.5
}

# Movement step along screw path
stepMM=0.5

# Minimum screw length
minLengthMM=18

# Allowed direction variation
directionConeDegLR=35
directionConeDegSI=20

# Number of directions tested
directionSamplesLR=13
directionSamplesSI=9

# Screw scoring weights
wDT=5.0
wLen=1.5
wTilt=0.5

# Segmentation label names
labelMap={5:"L1",4:"L2",3:"L3",2:"L4",1:"L5"}

# Load CT or segmentation file
def loadNifti(path):
    nii=nib.load(path)
    return nii.get_fdata(), nii.header.get_zooms(), nii.affine

ct,spacingCT,affineCT=loadNifti(ctPath)
seg,spacing,affine=loadNifti(segPath)

# Keep only real vertebra components
def getValidLabels(seg):
    valid=[]
    for labelVal in range(1,6):
        mask=(seg==labelVal)
        if np.sum(mask)==0:
            continue
        labeled,nc=cc_label(mask)
        sizes=np.bincount(labeled.ravel())
        sizes[0]=0
        largest=np.argmax(sizes)
        component=(labeled==largest)
        if np.sum(component)>voxelThreshold:
            valid.append((labelVal,component))
    return valid

validSegments=getValidLabels(seg)

# Find vertebra orientation (Up-Down, Left-Right, Front-Back)
def computeStableFrame(mask,affine):
    coords=np.argwhere(mask)
    coordsWorld=nib.affines.apply_affine(affine,coords)
    centroid=coordsWorld.mean(axis=0)
    pca=PCA(n_components=3)
    pca.fit(coordsWorld-centroid)
    axes=pca.components_
    worldZ=np.array([0,0,1])
    siAxis=axes[np.argmax(np.abs(axes@worldZ))]
    if np.dot(siAxis,worldZ)<0:
        siAxis=-siAxis
    tempAxis=axes[np.argmin(np.abs(axes@worldZ))]
    lrAxis=tempAxis-np.dot(tempAxis,siAxis)*siAxis
    lrAxis/=np.linalg.norm(lrAxis)
    apAxis=np.cross(siAxis,lrAxis)
    apAxis/=np.linalg.norm(apAxis)
    return centroid,np.vstack([siAxis,lrAxis,apAxis])

# Distance from bone boundary
# Big value = thicker bone = safer
def computeDistance(mask):
    return distance_transform_edt(mask,sampling=spacing)

# Find pedicle centers on left and right
def pedicleCenters(mask,dist,centroid,axes,affine):
    coords=np.argwhere(mask)
    coordsWorld=nib.affines.apply_affine(affine,coords)
    siAxis,lrAxis,apAxis=axes
    rel=coordsWorld-centroid
    siVals=rel@siAxis
    lrVals=rel@lrAxis
    apVals=rel@apAxis
    midMask=np.abs(siVals)<np.percentile(np.abs(siVals),35)
    posteriorMask=apVals<np.percentile(apVals,40)
    leftMask=lrVals<0
    rightMask=lrVals>0
    leftCoords=coords[midMask & posteriorMask & leftMask]
    rightCoords=coords[midMask & posteriorMask & rightMask]
    if len(leftCoords)<30 or len(rightCoords)<30:
        return None,None
    lVox=leftCoords[np.argmax(dist[leftCoords[:,0],leftCoords[:,1],leftCoords[:,2]])]
    rVox=rightCoords[np.argmax(dist[rightCoords[:,0],rightCoords[:,1],rightCoords[:,2]])]
    lMM=nib.affines.apply_affine(affine,lVox)
    rMM=nib.affines.apply_affine(affine,rVox)
    return lMM,rMM

# Find entry point on back surface
def findEntry(center,axes,maskFloat,affine):
    siAxis,lrAxis,apAxis=axes
    direction=-apAxis
    invAff=np.linalg.inv(affine)
    p=center.copy()
    for _ in range(300):
        vox=nib.affines.apply_affine(invAff,p)
        if any(v<0 or v>=s-1 for v,s in zip(vox,maskFloat.shape)):
            break
        val=map_coordinates(maskFloat,[[vox[0]],[vox[1]],[vox[2]]],order=1)[0]
        if val<0.5:
            break
        p+=direction*stepMM
    return p+apAxis*1.0

# Check if screw cylinder stays inside bone
def cylinderSafe(p,d,radius,maskFloat,affine):
    invAff=np.linalg.inv(affine)
    for angle in np.linspace(0,2*np.pi,8,endpoint=False):
        offset=radius*(np.cos(angle)*np.cross(d,[0,0,1])+np.sin(angle)*np.cross(d,[1,0,0]))
        testPoint=p+offset
        vox=nib.affines.apply_affine(invAff,testPoint)
        if any(v<0 or v>=s-1 for v,s in zip(vox,maskFloat.shape)):
            return False
        val=map_coordinates(maskFloat,[[vox[0]],[vox[1]],[vox[2]]],order=1)[0]
        if val<0.5:
            return False
    return True

# Test one screw path
def evaluate(entry,direction,maskFloat,dist,affine,radius,axes):
    d=direction/np.linalg.norm(direction)
    invAff=np.linalg.inv(affine)
    t=0
    minDT=999
    while True:
        if t<5:
            t+=stepMM
            continue
        p=entry+d*t
        vox=nib.affines.apply_affine(invAff,p)
        if any(v<0 or v>=s-1 for v,s in zip(vox,maskFloat.shape)):
            break
        maskVal=map_coordinates(maskFloat,[[vox[0]],[vox[1]],[vox[2]]],order=1)[0]
        if maskVal<0.5:
            break
        dtVal=map_coordinates(dist,[[vox[0]],[vox[1]],[vox[2]]],order=1)[0]
        minDT=min(minDT,dtVal)
        if not cylinderSafe(p,d,radius,maskFloat,affine):
            break
        t+=stepMM
    if t<minLengthMM:
        return None
    if minDT-radius<=0:
        return None
    siAxis=axes[0]
    tilt=abs(np.dot(d,siAxis))
    score=wDT*minDT + wLen*(t/10) - wTilt*tilt
    return score,t,minDT,p,d

# Try many diameters and directions
# Keep best safe screw
def optimize(center,axes,maskFloat,dist,affine,diameters):
    siAxis,lrAxis,apAxis=axes
    entry=findEntry(center,axes,maskFloat,affine)
    best=None
    for diam in diameters:
        radius=diam/2
        for lrAng in np.linspace(-directionConeDegLR,directionConeDegLR,directionSamplesLR):
            for siAng in np.linspace(-directionConeDegSI,directionConeDegSI,directionSamplesSI):
                direction=apAxis+np.tan(np.deg2rad(lrAng))*lrAxis+np.tan(np.deg2rad(siAng))*siAxis
                r=evaluate(entry,direction,maskFloat,dist,affine,radius,axes)
                if r is None:
                    continue
                score,length,minDT,tip,d=r
                if best is None or score>best[0]:
                    best=(score,entry,tip,length,minDT,diam)
    return best

# Store results for later use
resultsList=[]

print("\nFINAL ROBUST PEDICLE SCREW PLANNER:\n")

for labelVal,mask in validSegments:
    name=labelMap.get(labelVal,str(labelVal))
    maxDiam=maxDiameterPerLevel[name]
    diameters=[d for d in globalDiameters if d<=maxDiam]
    print("==============")
    print(name)
    print("==============")
    print("Tested Diameters:",diameters)
    centroid,axes=computeStableFrame(mask,affine)
    dist=computeDistance(mask)
    maskFloat=mask.astype(np.float32)
    lCenter,rCenter=pedicleCenters(mask,dist,centroid,axes,affine)
    for side,center in [("Left",lCenter),("Right",rCenter)]:
        result=optimize(center,axes,maskFloat,dist,affine,diameters)
        if result is None:
            print(f"{side}: NO SAFE PATH\n")
            continue
        score,entry,tip,length,minDT,diam=result
        resultsList.append({
            "vertebra":name,
            "side":side,
            "entry":entry,
            "tip":tip,
            "diameter":diam
        })
        print(f"{side} Screw Found")
        print("Diameter:",diam,"mm")
        print("Length:",round(length,1),"mm")
        print("Safety Margin:",round(minDT-diam/2,2),"mm")
        print()


FINAL ROBUST PEDICLE SCREW PLANNER:

L5
Tested Diameters: [8.5, 7.5, 7.0, 6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 5.5 mm
Length: 32.0 mm
Safety Margin: 0.1 mm

Right Screw Found
Diameter: 8.5 mm
Length: 36.5 mm
Safety Margin: 0.01 mm

L4
Tested Diameters: [8.5, 7.5, 7.0, 6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 7.5 mm
Length: 31.5 mm
Safety Margin: 0.29 mm

Right Screw Found
Diameter: 7.5 mm
Length: 32.0 mm
Safety Margin: 0.41 mm

L3
Tested Diameters: [7.5, 7.0, 6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 7.5 mm
Length: 29.5 mm
Safety Margin: 0.24 mm

Right Screw Found
Diameter: 7.5 mm
Length: 25.0 mm
Safety Margin: 0.19 mm

L2
Tested Diameters: [7.0, 6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 7.0 mm
Length: 19.0 mm
Safety Margin: 0.65 mm

Right Screw Found
Diameter: 7.0 mm
Length: 28.0 mm
Safety Margin: 0.22 mm

L1
Tested Diameters: [6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 6.5 mm
Length: 41.0 mm
Safety Margin: 0.39 mm

Right Screw F

***Visualization with TotalSegmentator***

In [10]:
import os

# Activate TotalSegmentator student license
# This allows the segmentation tool to run
print("Activating TotalSegmentator Student License...")
os.environ["TOTALSEG_LICENSE"] = "aca_V4E9OLA5VBGIQO"
print("License Activated")

# Run lumbar vertebra segmentation
# Input = CT scan
# Output = segmented vertebra files
print("Running TotalSegmentator Lumbar Vertebra Segmentation...")

inputCT="/content/drive/MyDrive/SpineData/case_002/case_0000.nii"
outputFolder="/content/totalseg_out"

# Remove old results if they exist
!rm -rf "$outputFolder"

# Create output folder
!mkdir -p "$outputFolder"

# Run TotalSegmentator
# This extracts vertebrae L1 to L5
!TotalSegmentator \
-i "$inputCT" \
-o "$outputFolder" \
-ta total \
-rs vertebrae_L1 vertebrae_L2 vertebrae_L3 vertebrae_L4 vertebrae_L5

print("Segmentation Completed")

Activating TotalSegmentator Student License...
License Activated
Running TotalSegmentator Lumbar Vertebra Segmentation...
No GPU detected. Running on CPU. This can be very slow. The '--fast' or the `--roi_subset` option can help to reduce runtime.

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
Resampling...
  Resampled in 2.75s
Predicting...
100% 12/12 [00:33<00:00,  2.81s/it]
  Predicted in 50.39s
Resampling...
  cropping from (366, 424, 283) to (82, 99, 139)
Resampling...
  Resampled in 0.00s
Predicting part 1 of 1 ...
100% 2/2 [00:44<00:00, 22.35s/it]
  Predicted in 69.07s
Resampling...
Saving segmentations...
  Saved in 9.71s
Segmentation Completed


In [11]:
import nibabel as nib
import numpy as np
from skimage.measure import marching_cubes

# Build a smooth 3D surface of the vertebrae
print("Building Smooth Vertebra Surface Mesh...")

totalsegFolder="/content/totalseg_out"

# Load segmented vertebrae L1 to L5
L1=nib.load(totalsegFolder+"/vertebrae_L1.nii.gz").get_fdata()
L2=nib.load(totalsegFolder+"/vertebrae_L2.nii.gz").get_fdata()
L3=nib.load(totalsegFolder+"/vertebrae_L3.nii.gz").get_fdata()
L4=nib.load(totalsegFolder+"/vertebrae_L4.nii.gz").get_fdata()
L5=nib.load(totalsegFolder+"/vertebrae_L5.nii.gz").get_fdata()

# Combine all vertebrae into one mask
vertebraMask=L1+L2+L3+L4+L5

# Create a smooth surface mesh from the mask
verts,faces,_,_=marching_cubes(vertebraMask,level=0.5)

# Convert mesh points from voxel space to real world coordinates
vertsWorld=nib.affines.apply_affine(affine,verts)

print("Mesh Ready")

Building Smooth Vertebra Surface Mesh...
Mesh Ready


In [12]:
import plotly.graph_objects as go
import numpy as np

# Create 3D visualization of vertebrae and screws
print("Creating Surgical-Grade Visualization...")

# Build cylindrical screw model
def createCylinder(entry,tip,diameter,resolution=40):
    entry=np.array(entry)
    tip=np.array(tip)
    direction=tip-entry
    length=np.linalg.norm(direction)
    direction=direction/length
    if abs(direction[2])<0.9:
        v=np.cross(direction,[0,0,1])
    else:
        v=np.cross(direction,[0,1,0])
    v=v/np.linalg.norm(v)
    w=np.cross(direction,v)
    radius=diameter/2
    theta=np.linspace(0,2*np.pi,resolution)
    z=np.linspace(0,length,30)
    theta,z=np.meshgrid(theta,z)
    x=radius*np.cos(theta)
    y=radius*np.sin(theta)
    X=entry[0]+direction[0]*z + v[0]*x + w[0]*y
    Y=entry[1]+direction[1]*z + v[1]*x + w[1]*y
    Z=entry[2]+direction[2]*z + v[2]*x + w[2]*y
    return X,Y,Z

fig=go.Figure()

# Add vertebra surface
fig.add_trace(go.Mesh3d(
x=vertsWorld[:,0],
y=vertsWorld[:,1],
z=vertsWorld[:,2],
i=faces[:,0],
j=faces[:,1],
k=faces[:,2],
opacity=0.25,
color='lightgray',
name="Lumbar Vertebrae"
))

# Draw screws
for r in resultsList:
    X,Y,Z=createCylinder(r["entry"],r["tip"],r["diameter"])
    fig.add_trace(go.Surface(
        x=X,
        y=Y,
        z=Z,
        showscale=False,
        opacity=1
    ))

# Show entry points
for r in resultsList:
    entry=r["entry"]
    fig.add_trace(go.Scatter3d(
        x=[entry[0]],
        y=[entry[1]],
        z=[entry[2]],
        mode='markers',
        marker=dict(size=5,color='green'),
        showlegend=False
    ))

# Display settings
fig.update_layout(
title="TotalSegmentator + Pedicle Screw Planner",
scene=dict(aspectmode='data'),
height=900
)

fig.show()

Creating Surgical-Grade Visualization...


***Visualization with CADSWholeBodyCTSeg***

# ***Phase 2 - AI Based Trajectory Optimization***